# 04. RL fine-tuning

Stage 4 of `docs/codeagents/research_plan.md` (§6.3, §7, §11 stage 4): the base model (`runs/base/best.pt`, π_ref) is fine-tuned with RLOO + exact KL to π_ref against the proxy RM (`runs/rm_proxy/best.pt`); the gold RM (`runs/rm_gold/best.pt`) is used for evaluation only. Run `02_baseline` and `03_reward_model` first.

- `rl_ratio_b0.1`: R1, reward = E[played ratio], β = 0.1. Expected hacking (plan §12): shorter, more popular and repeated tracks, proxy reward growing faster than gold.
- `rl_debiased_b0.1`: R4, ratio minus the train mean ratio of the track-length bin, which removes the advantage of short tracks.

Every `rl.eval_every` steps (and at step 0 = π_ref) the policy is evaluated on the val day: NDCG / Recall, proxy and gold reward of the top-10, its composition (share of history tracks, mean length, popularity, Gini) and the KL to π_ref. Logic lives in `recsys_rl.rl` and `recsys_rl.rewards`; the same run from a terminal: `uv run yambda-train-rl run_name=rl_ratio_b0.1`. A run resumes from `runs/<run_name>/last.pt` after a disconnect. Tables and plots for the report go to `results/stage4/` on Drive.

In [ ]:
# Bootstrap: each Colab notebook runs in its own runtime, so repeat the setup from 00.
# Data must already be on Drive (00_colab_setup.ipynb downloads it). Locally: cd to repo root.
import os
import sys
from pathlib import Path

if "google.colab" in sys.modules:
    from google.colab import drive, userdata

    drive.mount("/content/drive")
    os.environ["RECSYS_DATA_ROOT"] = "/content/drive/MyDrive/tlab-recsys"
    token = (userdata.get("GH_TOKEN") or "").strip()
    if not token:
        raise RuntimeError("Colab secret GH_TOKEN is empty, see notebooks/README.md")
    repo_url = f"https://{token}@github.com/miont/tlab-2026-recsys-task2.git"
    !git clone -q {repo_url} /content/repo || git -C /content/repo pull -q
    if not Path("/content/repo/pyproject.toml").exists():
        raise RuntimeError("git clone failed: check that GH_TOKEN can read the repository")
    %cd /content/repo
    !pip install -q uv
    !uv pip install --system -q -e .
    sys.path.insert(0, "/content/repo/src")  # the editable .pth is only read at startup
elif not Path("configs").exists():
    os.chdir(Path.cwd().parent)

In [ ]:
import matplotlib.pyplot as plt
import polars as pl
import torch

from recsys_rl import evaluate
from recsys_rl.config import load_config, parse_overrides
from recsys_rl.data import prepare
from recsys_rl.model import load_model
from recsys_rl.rl import train_rl
from recsys_rl.train import amp_dtype
from recsys_rl.utils import get_device, save_figure

pl.Config.set_tbl_rows(40)
pl.Config.set_tbl_cols(-1)
pl.Config.set_tbl_width_chars(250)
pl.Config.set_float_precision(4)

device = get_device()
print(device, torch.cuda.get_device_name() if device.type == "cuda" else "")
cfg = load_config("configs/base.yaml")
prepared = prepare(cfg)
RESULTS = cfg.results_dir / "stage4"  # CSV / PNG for the report, on Drive in Colab
RESULTS.mkdir(parents=True, exist_ok=True)

## Training

Defaults are in `RLConfig` (`src/recsys_rl/config.py`): 3000 steps of 64 contexts × 16 sampled candidates, lr 5e-5, evaluation every 250 steps. Overrides use the CLI syntax, e.g. `"rl.kl_beta=0.05"`.

In [ ]:
RUNS = {
    "rl_ratio_b0.1": ["rl.reward=ratio", "rl.kl_beta=0.1"],
    "rl_debiased_b0.1": ["rl.reward=debiased_ratio", "rl.kl_beta=0.1"],
}


def run(run_name: str, *overrides: str):
    run_cfg = load_config(
        "configs/base.yaml", **parse_overrides([f"run_name={run_name}", *overrides])
    )
    return load_model(train_rl(run_cfg, prepared, device), device)


policies = {name: run(name, *overrides) for name, overrides in RUNS.items()}

## Trajectories on the val day

Step 0 is π_ref. Proxy reward growing while gold reward stalls or falls is the over-optimisation signal (Gao et al., 2022); the composition panels show what the policy moves towards.

In [ ]:
logs = pl.concat(
    [
        pl.read_ndjson(cfg.run_dir.parent / name / "log.jsonl").with_columns(run=pl.lit(name))
        for name in RUNS
    ],
    how="diagonal_relaxed",
)
logs.write_csv(RESULTS / "trajectories.csv")

PANELS = [
    ("val_proxy_reward@10", "proxy reward@10"),
    ("val_gold_reward@10", "gold reward@10"),
    ("val_ndcg@10", "NDCG@10"),
    ("val_kl_ref", "KL(π || π_ref)"),
    ("val_repeat@10", "share of history tracks@10"),
    ("val_length@10", "mean track length@10, s"),
    ("val_log_count@10", "mean log10 train count@10"),
    ("val_coverage@10", "coverage@10"),
]
fig, axes = plt.subplots(2, 4, figsize=(20, 8))
for ax, (col, title) in zip(axes.ravel(), PANELS, strict=True):
    for name in RUNS:
        log = logs.filter(pl.col("run") == name)
        ax.plot(log["step"], log[col], marker="o", label=name)
    ax.set(xlabel="step", title=title)
axes[0, 0].legend()
save_figure(fig, RESULTS / "trajectories.png")
plt.show()

logs.select(
    "run", "step", *[col for col, _ in PANELS], "val_recall@100", "train_reward", "train_entropy"
)

## Final policies vs π_ref (val and test)

The same protocol as stage 2, with and without the repeat filter.

In [ ]:
COLUMNS = [
    "split",
    "method",
    "exclude_seen",
    "ndcg@10",
    "recall@10",
    "recall@100",
    "paper_ndcg@10",
    "paper_recall@100",
    "coverage@10",
]
base = load_model(cfg.run_dir.parent / cfg.rl.base_run / "best.pt", device)
results = evaluate.results_table(
    prepared,
    cfg.data,
    models={"gpt2_base": base, **policies},
    amp_dtype=amp_dtype(device, cfg.rl.amp),
)
results.write_csv(RESULTS / "results.csv")
results.select(COLUMNS)